In [1]:
import torch
from diffusion.approaches.matching.alphas_betas import LinearAlpha, LinearBeta
from diffusion.approaches.matching.prob_paths import GaussianCondProbPath
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.data.mnist_sampleable import MNISTSampleable
from diffusion.architectures.res_unet import ResUnet
from diffusion.architectures.classifier.mnist_classifier import SimpleCNN

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable(train=True)

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=(1, 32, 32),
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

encoder = SimpleCNN(sampeable.num_classes).to(device)
encoder.load_state_dict(torch.load("./models/classifier.pt"))

trainer = FlowTrainer(
    path=path,
    backbone=backbone,
    encoder=encoder,
    num_classes=sampeable.num_classes,
)

In [4]:
trainer.train(
    num_epochs=5,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-07 17:47:13,561 - flow-matching - INFO - Training model with size: 2.734 MiB
100%|██████████| 9/9 [00:13<00:00,  1.54s/it]
2025-10-07 17:49:08,167 - flow-matching - INFO - ['kid: 25991.442188', 'precision: 0.710200', 'recall: 0.461400', 'f1: 0.559201']
100%|██████████| 9/9 [00:13<00:00,  1.54s/it]
2025-10-07 17:51:00,329 - flow-matching - INFO - ['kid: 21242.871875', 'precision: 0.472200', 'recall: 0.703000', 'f1: 0.564588']
100%|██████████| 9/9 [00:13<00:00,  1.55s/it]
2025-10-07 17:52:52,561 - flow-matching - INFO - ['kid: 24888.059375', 'precision: 0.407400', 'recall: 0.845900', 'f1: 0.549753']
100%|██████████| 9/9 [00:13<00:00,  1.55s/it]
2025-10-07 17:54:44,883 - flow-matching - INFO - ['kid: 67127.856250', 'precision: 0.364300', 'recall: 0.869400', 'f1: 0.513423']
100%|██████████| 9/9 [00:13<00:00,  1.54s/it]
2025-10-07 17:56:37,130 - flow-matching - INFO - ['kid: 38312.518750', 'precision: 0.521400', 'recall: 0.796600', 'f1: 0.630191']


In [6]:
torch.save(backbone.state_dict(), "./models/backbone_flow.pt")